# UC1 — PCB (Printed Circuit Board): 3. Auto Mask Placement (Testcase Preparation)

Generation needs, for every image it will produce, a **(clean image, defect mask,
anomaly type)** triple. Rather than hand-draw masks, we place the real training
submasks onto clean images automatically — this is **Auto Mask Placement (AMP)** — and
emit a `testcase.jsonl` describing each generation.

The whole step is wrapped by **`prep_testcase.sh`**, which runs:
`validate → allocate samples → build AMP sample list → AMP → build JSONL → verify`.

> **How commands run in this tutorial.** All pipeline steps run inside the
> `cosmos-predict2` conda environment. In a notebook cell we prefix shell
> commands with `conda run -n cosmos-predict2` (add `--live-stream` to stream
> logs live). If you prefer, open a JupyterLab **Terminal**, run
> `conda activate cosmos-predict2` once, and paste the same commands without
> the `conda run` prefix.
>
> If that environment does not exist yet, build it first with the top-level
> [tutorial/notebooks/0-setup-cuda128.ipynb](../../0-setup-cuda128.ipynb) — see
> the prerequisite note below.

## 3.0 Set the project root

In [ ]:
# Resolve the repository root (the folder containing pyproject.toml) and cd into it,
# so every relative path below (datasets/, checkpoints/, results/, scripts/) resolves.
import os
d = os.getcwd()
while d != "/" and not os.path.exists(os.path.join(d, "pyproject.toml")):
    d = os.path.dirname(d)
LOCAL_PROJECT_DIR = d
os.chdir(LOCAL_PROJECT_DIR)
# Pipeline scripts read the finetuned models & write outputs under the repo root.
os.environ.setdefault("IMAGINAIRE_OUTPUT_ROOT", "./results")
print("Project root:", LOCAL_PROJECT_DIR)

## 3.1 How masks are placed for UC1

AMP routes each defect by its `spatial_dependency` (from `defect_spec.jsonl`):

- `free` → whole-image ROI (defect can appear anywhere)
- `cad` → CAD-mask ROI (defect tied to component/pad geometry)
- `text` → text-prompt ROI via Qwen-VL + SAM2 (defect confined to a described region)

For **UC1** every defect is **`cad`**:

**`cad` → cad2roi.** PCB defects are *location-dependent*: a bridge can only occur across IC pins, missing/excess solder only on a component's solder pads. The legal region for each placement is therefore derived from a **CAD mask** (`<TEXTURE>/cad_mask/<stem>.png`) together with `semantic_segmentation_labels.json`. `prep_testcase.sh` routes every `cad` defect through `cad2roi` automatically.

## 3.2 Build the testcase

`--num-sdg` is the total number of synthetic images to plan (spread across defect
types). This writes `ag_inference/UC1_pcb/testcase.jsonl` plus the AMP artifacts under
`ag_inference/UC1_pcb/amp/`.

In [ ]:
!conda run -n cosmos-predict2 bash scripts/utilities/prep_testcase.sh \
    --name UC1_pcb \
    --num-sdg 6 \
    --dataset-dir datasets/UC1_pcb \
    --amp-output-dir ag_inference/UC1_pcb/amp \
    --output-jsonl ag_inference/UC1_pcb/testcase.jsonl \
    --defect-spec datasets/UC1_pcb/defect_spec.jsonl \
    --mode inference

## 3.2b (Self-training only) Build the validation testcase

If you plan to **train it yourself** (notebook 2, §2.2), the training config's
`dataloader_val` reads a validation test set at
`ag_inference/UC1_pcb_validation/testcase.jsonl`. Build it here with
**`--mode validation`**, which places **every training mask once** (≥1 sample per
defect → a meaningful validation KPI). `--num-sdg` is the total training-mask
count. **Skip this if you only use the released checkpoint** (notebook 2, §2.3).

In [ ]:
# (Self-training only) Build the validation test set that dataloader_val reads.
# Count training masks exactly as validate_dataset.py does (image files under each
# anomaly_image type's mask/ folder), so --mode validation places every mask once.
from pathlib import Path
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}
NUM_VAL = 0
for texture in Path("datasets/UC1_pcb").iterdir():
    img_root, mask_root = texture / "anomaly_image", texture / "mask"
    if not (img_root.is_dir() and mask_root.is_dir()):
        continue
    for atype in img_root.iterdir():                     # types come from anomaly_image/
        mdir = mask_root / atype.name
        if atype.is_dir() and mdir.is_dir():
            NUM_VAL += sum(1 for p in mdir.iterdir() if p.suffix.lower() in IMG_EXTS)
print("Total training masks:", NUM_VAL)
!conda run -n cosmos-predict2 bash scripts/utilities/prep_testcase.sh \
    --name UC1_pcb_validation \
    --num-sdg {NUM_VAL} \
    --dataset-dir datasets/UC1_pcb \
    --amp-output-dir ag_inference/UC1_pcb_validation/amp \
    --output-jsonl ag_inference/UC1_pcb_validation/testcase.jsonl \
    --defect-spec datasets/UC1_pcb/defect_spec.jsonl \
    --mode validation

## 3.3 Inspect the testcase

Each line is one generation task. Note `image_filename` (clean canvas), `mask_filename` (the AMP-placed mask), `anomaly_type`, and the diffusion/crop parameters.

In [ ]:
import json
p = "ag_inference/UC1_pcb/testcase.jsonl"
lines = open(p).read().splitlines()
print(f"{len(lines)} entries in {p}\n\nFirst entry:")
print(json.dumps(json.loads(lines[0]), indent=2))

## 3.4 Example: an AMP-placed mask

Below is one `IC+bridge` submask after AMP placed it onto a clean PCB image
(left: the placed binary mask; right: the mask overlaid on the clean image), produced
by the command above.

| Placed mask | Overlay on clean image |
|---|---|
| ![](assets/amp/IC+bridge_placed_mask.png) | ![](assets/amp/IC+bridge_overlay.png) |


## Next Step

Proceed to [4-generation.ipynb](./4-generation.ipynb).